# 📖 Lab 4: Scaling to 1M Requests/Second (Deep Dive)

**Non-functional requirement:** *Handle 1M requests/second across 100M daily active users.*

A single Redis instance handles ~100K ops/sec. Our Token Bucket Lua script is a single operation per check, but at 1M req/s we need 10+ Redis instances. The challenge: all requests from the same client **must hit the same shard**.

## 🏗️ Architecture — Before (Single Redis)

```
Gateway A ──┐
Gateway B ──┼──> Redis (single instance, ~100K ops/sec) 💀 at 1M req/s
Gateway C ──┘
```

## 🏗️ Architecture — After (Sharded Redis)

```
Gateway A ──┐      hash(clientId)     ┌──> Redis Shard 1 (~100K ops/s)
Gateway B ──┼──> Routing Logic ──────>├──> Redis Shard 2 (~100K ops/s)
Gateway C ──┤                         ├──> Redis Shard 3 (~100K ops/s)
Gateway D ──┘                         └──> ...Shard N
                                       10 shards = 1M ops/s ✅
```

## Learning Objectives

- Understand why single Redis fails at scale (math it out)
- Implement consistent hashing to route clients to shards
- Prove that all requests for one client always hit the same shard
- Simulate multi-shard rate limiting with correct enforcement
- Understand Redis Cluster vs manual sharding

## 🛠️ Setup

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

Select the **"Rate Limiter (Python)"** kernel.

In [ ]:
import hashlib
import time
import redis
from collections import Counter

redis_client = redis.Redis(host="localhost", port=6381, decode_responses=True)
redis_client.flushdb()
print(f"✅ Redis: connected")

## 📊 The Math: Why Single Redis Fails

Let's do the capacity planning math to understand the bottleneck.

In [ ]:
TARGET_RPS = 1_000_000
REDIS_OPS_PER_SEC = 100_000  # realistic for Lua script operations
OPS_PER_CHECK = 1             # our Lua script is 1 atomic operation

checks_per_second = TARGET_RPS * OPS_PER_CHECK
shards_needed = checks_per_second / REDIS_OPS_PER_SEC

print(f"📊 Capacity Planning:\n")
print(f"  Target:             {TARGET_RPS:>12,} requests/sec")
print(f"  Redis capacity:     {REDIS_OPS_PER_SEC:>12,} ops/sec (per instance)")
print(f"  Ops per RL check:   {OPS_PER_CHECK:>12}")
print(f"  Total ops needed:   {checks_per_second:>12,} ops/sec")
print(f"  Shards needed:      {shards_needed:>12.0f}")
print(f"\n  💡 We need ~{shards_needed:.0f} Redis shards to handle {TARGET_RPS:,} req/s")
print(f"     A single Redis instance would be {checks_per_second / REDIS_OPS_PER_SEC:.0f}x overloaded!")

## 🔧 Consistent Hashing: Routing Clients to Shards

The critical requirement: **all requests from client X must always hit the same shard**. If Alice's requests split across shards, her rate limit state is divided and useless.

We use consistent hashing: `hash(clientId) % num_shards → shard index`.

In [ ]:
NUM_SHARDS = 10

def get_shard(client_id: str, num_shards: int = NUM_SHARDS) -> int:
    """Deterministic shard routing using consistent hashing."""
    hash_val = int(hashlib.md5(client_id.encode()).hexdigest(), 16)
    return hash_val % num_shards


# Prove: same client ALWAYS maps to same shard
print("🔧 Consistent Hashing — same client → same shard:\n")
test_clients = ["alice", "bob", "charlie", "dave", "eve",
                "user_12345", "192.168.1.1", "api_key_abc"]

for client in test_clients:
    shard = get_shard(client)
    # Call it 3 times to prove determinism
    shards = [get_shard(client) for _ in range(3)]
    consistent = "✅" if len(set(shards)) == 1 else "❌"
    print(f"  {client:<20} → shard {shard}  (deterministic: {consistent})")

# Check distribution across shards
print(f"\n📊 Distribution of 10,000 random users across {NUM_SHARDS} shards:\n")
shard_counts = Counter()
for i in range(10_000):
    shard_counts[get_shard(f"user_{i}")] += 1

for shard_id in sorted(shard_counts):
    count = shard_counts[shard_id]
    bar = "█" * (count // 50)
    print(f"  Shard {shard_id}: {count:>5} users  {bar}")

min_count = min(shard_counts.values())
max_count = max(shard_counts.values())
balance = min_count / max_count * 100
print(f"\n  Balance: {balance:.0f}% (min={min_count}, max={max_count})")
print(f"  💡 Good hash functions distribute keys ~evenly across shards.")

## 🧪 Simulating Sharded Rate Limiting

We'll simulate multiple Redis shards (using key prefixes on our single Redis for the demo) and prove that rate limiting works correctly even when sharded.

In [ ]:
TOKEN_BUCKET_LUA = """
local key = KEYS[1]
local capacity = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])
local ttl = tonumber(ARGV[4])

local tokens = tonumber(redis.call('HGET', key, 'tokens') or capacity)
local last_refill = tonumber(redis.call('HGET', key, 'last_refill') or now)

local elapsed = now - last_refill
tokens = math.min(capacity, tokens + elapsed * refill_rate)

local allowed = 0
local remaining = math.floor(tokens)

if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
    remaining = math.floor(tokens)
end

redis.call('HSET', key, 'tokens', tostring(tokens))
redis.call('HSET', key, 'last_refill', tostring(now))
redis.call('EXPIRE', key, ttl)

return {allowed, remaining}
"""

token_bucket_script = redis_client.register_script(TOKEN_BUCKET_LUA)

CAPACITY = 5
REFILL_RATE = 1.0
redis_client.flushdb()


def sharded_rate_check(client_id: str) -> dict:
    """Rate limit check with shard routing."""
    shard = get_shard(client_id)
    # In production: route to different Redis instances
    # Here: simulate with key prefix
    key = f"shard:{shard}:rl:{client_id}"
    now = time.time()

    result = token_bucket_script(keys=[key], args=[CAPACITY, REFILL_RATE, now, 3600])
    return {"allowed": bool(result[0]), "remaining": result[1], "shard": shard}


# Test: Alice sends 8 requests from different "gateways"
# All must hit the same shard
print(f"🧪 Sharded Rate Limiting — Alice from multiple gateways:\n")
print(f"  Config: capacity={CAPACITY}, refill={REFILL_RATE}/sec, shards={NUM_SHARDS}\n")

alice_shard = get_shard("alice")
print(f"  Alice routes to: shard {alice_shard}\n")

gateways = ["GW-A", "GW-B", "GW-C", "GW-A", "GW-B", "GW-C", "GW-A", "GW-B"]
for i, gw in enumerate(gateways):
    result = sharded_rate_check("alice")
    status = "✅" if result["allowed"] else "❌ 429"
    print(f"  {gw}: Request {i+1} → shard {result['shard']} → {status} (remaining: {result['remaining']})")

print(f"\n  ✅ All 8 requests from different gateways hit shard {alice_shard}.")
print(f"     Rate limiting works correctly — exactly {CAPACITY} allowed, rest rejected.")

## 📊 Proving Load Distribution Across Shards

With many clients, requests should distribute evenly across shards. Let's simulate 1000 different users making requests and see how the load spreads.

In [ ]:
redis_client.flushdb()

NUM_USERS = 1000
REQUESTS_PER_USER = 3

shard_ops = Counter()  # how many ops hit each shard

for u in range(NUM_USERS):
    client_id = f"user_{u}"
    shard = get_shard(client_id)
    for _ in range(REQUESTS_PER_USER):
        sharded_rate_check(client_id)
        shard_ops[shard] += 1

total_ops = sum(shard_ops.values())

print(f"📊 Load distribution: {NUM_USERS} users × {REQUESTS_PER_USER} requests = {total_ops} total ops\n")
print(f"  {'Shard':<8} {'Ops':<8} {'% of total':<12} {'Bar'}")
print(f"  {'─'*50}")
for shard_id in sorted(shard_ops):
    ops = shard_ops[shard_id]
    pct = ops / total_ops * 100
    bar = "█" * int(pct * 2)
    print(f"  {shard_id:<8} {ops:<8} {pct:>5.1f}%       {bar}")

print(f"\n  Each shard handles ~{total_ops // NUM_SHARDS} ops ({100 / NUM_SHARDS:.0f}% of total)")
print(f"  At 1M req/s → each shard handles ~{1_000_000 // NUM_SHARDS:,} req/s ✅")

## 🧹 Cleanup

In [ ]:
redis_client.flushdb()
print("✅ Redis cleaned up.")

## ✅ Summary

### The Scaling Math

```
1M req/s ÷ 100K ops/sec per Redis = 10 shards needed
```

### Key Design Decisions

| Decision | Why |
|----------|-----|
| **Consistent hashing** | Same client always hits same shard → rate limit state stays intact |
| **Hash on client ID** | User ID for auth, IP for anonymous, API key for developers |
| **Even distribution** | Good hash function distributes ~evenly → no hot shards |

### Manual Sharding vs Redis Cluster

| | Manual Sharding | Redis Cluster |
|-|-----------------|---------------|
| **Routing** | Application builds hash → picks shard | Redis auto-routes via 16,384 hash slots |
| **Rebalancing** | Manual when adding/removing shards | Automatic slot migration |
| **Failover** | Application must handle | Built-in replica promotion |
| **Complexity** | Build it yourself | Built-in, but cluster mode has constraints (no cross-slot Lua) |

### Interview Guidance

- **Calculate the math:** show you understand throughput limits
- **Articulate the sharding requirement:** same client → same shard, always
- **Mention Redis Cluster** as the production solution
- **Call out hot-key risk:** one viral user shouldn't overload a shard (mitigate with local caches or rate limiting at the gateway level before Redis)